# Road maps

Explore arc length, lateral offsets and the limits of polyline curvature.

Run with the repository virtual environment. These are synthetic research experiments; units and assumptions are explicit below.

## Road families

The road surface is represented by a centerline and a constant width.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from autonomous_vehicles.simulation.geometry import Vec2
from autonomous_vehicles.simulation.roads import circular_road, curved_road, straight_road

roads = {"straight": straight_road(), "winding": curved_road(), "arc": circular_road()}

## Draw road boundaries

Boundaries are sampled at signed offsets from the centerline.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (name, road) in zip(axes, roads.items(), strict=True):
    for lateral in (-road.width / 2, 0, road.width / 2):
        samples = road.samples(1, lateral)
        ax.plot([p.x for p in samples], [p.y for p in samples])
    ax.set_title(f"{name}: {road.length:.1f} m")
    ax.set_aspect("equal", adjustable="datalim")
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
plt.tight_layout()
plt.show()

## Frenet-like coordinates

Projection returns arc length and signed distance. Corners and endpoints can be ambiguous.

In [ ]:
road = roads["winding"]
queries = [road.pose_at(s, lateral).position for s, lateral in ((20, 1), (50, -2), (80, 0.5))]
for point in queries:
    projection = road.project(point)
    recovered = road.pose_at(projection.s, projection.lateral)
    error = (recovered.position - point).norm
    print({"s": projection.s, "lateral": projection.lateral, "roundtrip_error": error})

## Curvature window

Polyline headings are discontinuous. A window estimates average curvature over distance.

In [ ]:
road = roads["arc"]
coordinates = np.linspace(3, road.length - 3, 80)
fig, ax = plt.subplots(figsize=(8, 3))
for window in (1, 2, 4):
    curvature = [road.curvature_at(float(s), window) for s in coordinates]
    ax.plot(coordinates, curvature, label=f"window={window} m")
ax.axhline(1 / 40, color="black", linestyle="--", label="Circle 1/r")
ax.set_xlabel("Arc length (m)")
ax.set_ylabel("Curvature (1/m)")
ax.legend()
plt.show()

## Finite extent

Projection clamps beyond endpoints. Road containment includes finite endpoint distance.

In [ ]:
road = roads["straight"]
points = [Vec2(-10, 0), Vec2(10, 0), Vec2(10, 5), Vec2(160, 0)]
checks = [(point, road.project(point).s, road.contains(point)) for point in points]
for point, arc, inside in checks:
    print({"point": point, "s": arc, "inside": inside})
assert road.contains(Vec2(10, 0))
assert not road.contains(Vec2(10, 5))

## Research record

Record the seed, configuration, dependency lock and observed failures with each experiment. Results here demonstrate the implemented model under its stated assumptions. They do not establish performance outside this synthetic distribution.